In [2]:
from dotenv import load_dotenv

from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings,
)

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import (
    RunnableParallel,
    RunnablePassthrough,
)
from langchain_core.output_parsers import StrOutputParser

import os
os.environ["LANGCHAIN_PROJECT"] = "RAG Chatbot"

load_dotenv()

# --------------------------------------------------
# Load PDF
# --------------------------------------------------

loader = PyPDFLoader("Conor_McGregor_Overview.pdf")
documents = loader.load()

# --------------------------------------------------
# Split PDF
# --------------------------------------------------

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
)

chunks = splitter.split_documents(documents)

# --------------------------------------------------
# Embeddings
# --------------------------------------------------

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

# --------------------------------------------------
# Vector Store
# --------------------------------------------------

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_db"
)

retriever = vector_db.as_retriever(
    search_kwargs={"k": 3}
)

# --------------------------------------------------
# LLM
# --------------------------------------------------

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,
)

# --------------------------------------------------
# Prompt
# --------------------------------------------------

prompt = ChatPromptTemplate.from_template(
    """
You are a helpful assistant.

Answer ONLY using the provided context.

If the answer is not available in the context, reply:

"I couldn't find that information in the PDF."

Context:
{context}

Question:
{question}

Answer:
"""
)

# --------------------------------------------------
# Format retrieved documents
# --------------------------------------------------

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# --------------------------------------------------
# Parallel Chain
# --------------------------------------------------

parallel_chain = RunnableParallel(
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
)

# --------------------------------------------------
# Complete RAG Chain
# --------------------------------------------------

rag_chain = (parallel_chain| prompt| llm| StrOutputParser())

# --------------------------------------------------
# Chat Loop
# --------------------------------------------------

print("=" * 60)
print("Simple LangChain RAG")
print("Type 'exit' to quit")
print("=" * 60)

while True:

    question = input("\nQuestion: ")

    if question.lower() == "exit":
        break

    answer = rag_chain.invoke(question)

    print("\nAnswer:\n")
    print(answer)

ModuleNotFoundError: No module named 'annotated_types'